In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load & basic checks
df = pd.read_csv('hamrobazaar_mobiles_cleaned.csv')
print(f"Dataset Shape: {df.shape}")
print(f"Price Stats:\n{df['Price_Cleaned'].describe()}")
print(f"Features dtypes:\n{df[['Location_City_Encoded', 'negotiable_encoded', 'Storage_GB_Corrected', 
                          'RAM_GB_Corrected', 'Screen_Size_Inches', 'Back_Camera_MP_Filtered', 
                          'Brand_Model_Encoded']].dtypes}")

# Quick split for consistency
features = [
    'Location_City_Encoded', 'negotiable_encoded',
    'Storage_GB_Corrected', 'RAM_GB_Corrected',
    'Screen_Size_Inches', 'Back_Camera_MP_Filtered',
    'Brand_Model_Encoded'
]
X = df[features].fillna(0)
y = df['Price_Cleaned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Dataset Shape: (652, 11)
Price Stats:
count       652.000000
mean      68826.792945
std       60201.043156
min        1000.000000
25%       19000.000000
50%       46994.500000
75%      105000.000000
max      230000.000000
Name: Price_Cleaned, dtype: float64
Features dtypes:
Location_City_Encoded        int64
negotiable_encoded           int64
Storage_GB_Corrected         int64
RAM_GB_Corrected             int64
Screen_Size_Inches         float64
Back_Camera_MP_Filtered      int64
Brand_Model_Encoded          int64
dtype: object
Train size: 521, Test size: 131


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import time
import warnings
warnings.filterwarnings('ignore')

# Load & enhanced features (add title length + basic scam flag for better prediction)
df = pd.read_csv('hamrobazaar_mobiles_cleaned.csv')
df['title_length'] = df['title'].str.len().fillna(0)
df['scam_flag'] = df['title'].str.contains('17 Pro Max|S25 Ultra|17 Air|16e', case=False, na=False).astype(int)

features = [
    'Location_City_Encoded', 'negotiable_encoded',
    'Storage_GB_Corrected', 'RAM_GB_Corrected',
    'Screen_Size_Inches', 'Back_Camera_MP_Filtered',
    'Brand_Model_Encoded', 'title_length', 'scam_flag'
]
X = df[features].fillna(0)
y = df['Price_Cleaned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Improved models (tamed XGBoost for small data, tuned RF)
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,  # Limit to avoid overfit on 652 rows
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

xgb = XGBRegressor(
    n_estimators=200,  # Fewer trees for small data
    learning_rate=0.05,  # Slower learning
    max_depth=6,  # Shallower trees
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

models = {'RandomForest': rf, 'XGBoost': xgb}
results = []

print("Improved XGBoost vs RandomForest (With Feature Engineering)\n" + "="*70)

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Predictions
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    
    # Metrics (fixed rounding)
    mae = mean_absolute_error(y_test, pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred_test))
    r2 = r2_score(y_test, pred_test)
    
    # CV score for stability
    cv_mae = -cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error').mean()
    
    results.append({
        'Model': name,
        'Test MAE': mae,
        'CV MAE': cv_mae,
        'Test RMSE': rmse,
        'Test R²': r2,
        'Train Time (s)': train_time
    })
    
    print(f"{name:12} → Test MAE: {mae:8,.0f} | CV MAE: {cv_mae:8,.0f} | RMSE: {rmse:8,.0f} | R²: {r2:.4f} | Time: {train_time:.2f}s")

# Results table (no rounding issues)
results_df = pd.DataFrame(results).round(2)
results_df = results_df.sort_values('Test MAE')
print("\n" + "="*70)
print("WINNER:", results_df.iloc[0]['Model'])
print(results_df[['Model', 'Test MAE', 'CV MAE', 'Test RMSE', 'Test R²', 'Train Time (s)']])

Improved XGBoost vs RandomForest (With Feature Engineering)
RandomForest → Test MAE:   15,734 | CV MAE:   16,151 | RMSE:   24,634 | R²: 0.8095 | Time: 0.54s
XGBoost      → Test MAE:   14,257 | CV MAE:   15,389 | RMSE:   21,819 | R²: 0.8505 | Time: 0.18s

WINNER: XGBoost
          Model  Test MAE    CV MAE  Test RMSE  Test R²  Train Time (s)
1       XGBoost  14257.41  15389.29   21818.87     0.85            0.18
0  RandomForest  15734.01  16150.72   24633.96     0.81            0.54


In [5]:
from sklearn.model_selection import GridSearchCV

# Assume XGBoost wins — tune it (swap to RF if needed)
param_grid = {
    'n_estimators': [150, 200, 250],
    'learning_rate': [0.03, 0.05, 0.07],
    'max_depth': [5, 6, 7],
    'min_child_weight': [3, 5]
}

xgb_base = XGBRegressor(subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1, verbosity=0)
grid = GridSearchCV(xgb_base, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best XGBoost Params:")
for k, v in grid.best_params_.items():
    print(f"{k}: {v}")
print(f"Best CV MAE: {-grid.best_score_:,.0f} NPR")

best_model = grid.best_estimator_
test_pred = best_model.predict(X_test)
final_mae = mean_absolute_error(y_test, test_pred)
print(f"Final Test MAE: {final_mae:,.0f} NPR")

Best XGBoost Params:
learning_rate: 0.05
max_depth: 7
min_child_weight: 3
n_estimators: 150
Best CV MAE: 15,821 NPR
Final Test MAE: 13,925 NPR


In [6]:
import joblib

# Retrain on full data (use XGBoost or swap to RF)
final_model = XGBRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=7,
    min_child_weight=3,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1
)
final_model.fit(X, y)  # X now includes new features
joblib.dump(final_model, 'nepal_phone_final_model.pkl')
print("Final model saved as 'nepal_phone_final_model.pkl'")

Final model saved as 'nepal_phone_final_model.pkl'
